In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from nbqol import path_to_git_root

In [2]:
ROOT = Path(path_to_git_root()) # automatically makes path to repo directory (wherever you cloned it)
print("Repository directory: ", ROOT)

QPCR = ROOT / "qPCR-validation" / "qpcr_data"
print("qPCR data directory: ", QPCR)

RNASEQ = ROOT / "qPCR-validation" / "bulkRNAseq_data"
print("RNA-Seq data directory: ", RNASEQ)

Repository directory:  /Users/timothy/Documents/EV-Resistance-analysis
qPCR data directory:  /Users/timothy/Documents/EV-Resistance-analysis/qPCR-validation/qpcr_data
RNA-Seq data directory:  /Users/timothy/Documents/EV-Resistance-analysis/qPCR-validation/bulkRNAseq_data


Load qPCR + RNAseq data

In [3]:
# qPCR
umuc1_ht8_qpcr_df = pd.read_csv(QPCR / "QPCR_6-12_UMUC1-HT8.csv")
rt112_2f2_qpcr_df = pd.read_csv(QPCR / "QPCR_6-15_RT112-2F2E-2F2L.csv")
v647_qpcr_df = pd.read_csv(QPCR / "QPCR_6-16_647V-1C2.csv")
umuc1_ht8_647v_1c2_crispr_df = pd.read_csv(QPCR / "QPCR_umuc1_ht8_647v_1c2_crispr.csv")
rt112_2f2_crispr_df = pd.read_csv(QPCR / "QPCR_rt112_2f2_crispr.csv")

redo_df = pd.read_csv("/Users/timothy/Downloads/qPCR Master Datasheet.xlsx - 6-30_ALL-samples_redo.csv")

combined_qPCR_df = pd.concat([umuc1_ht8_qpcr_df, rt112_2f2_qpcr_df, v647_qpcr_df, umuc1_ht8_647v_1c2_crispr_df, rt112_2f2_crispr_df, redo_df], ignore_index=True)


# RNA-Seq
umuc1_ht8_rnaseq_df = pd.read_csv(RNASEQ / "bulkRNA_UMUC1_parental_vs_resistant.csv")
umuc1_ht8_rnaseq_df['Sample'] = "UMUC1-HT8"

rt112_2f2_rnaseq_df = pd.read_csv(RNASEQ / "bulkRNA_RT112_parental_vs_resistant.csv")
rt112_2f2_rnaseq_df['Sample'] = "RT112-2F2"

v647_rnaseq_df = pd.read_csv(RNASEQ / "bulkRNA_647V_parental_vs_resistant.csv")
v647_rnaseq_df['Sample'] = "647V-1C2"

combined_RNAseq_df = pd.concat([umuc1_ht8_rnaseq_df, rt112_2f2_rnaseq_df, v647_rnaseq_df], ignore_index=True)

In [4]:
combined_qPCR_df

,Sample Name,Target Name,CT,Ct Mean,HIGHSD,NOAMP,EXPFAIL,Ct SD,MTP
0,UMUC1,ACTIN,18.327,18.252,N,N,N,NaN,NaN
1,UMUC1,ACTIN,18.177,18.252,N,N,N,NaN,NaN
2,UMUC1,EHD3,27.914,27.93,N,N,N,NaN,NaN
3,UMUC1,EHD3,27.945,27.93,N,N,N,NaN,NaN
4,UMUC1,CLIC4,24.1,24.09,N,N,N,NaN,NaN
...,...,...,...,...,...,...,...,...,...
1133,2F2-LATE,GJB1,36.231,34.268,Y,N,N,NaN,NaN
1134,UMUC1,TMCC2,32.399,32.475,N,N,N,NaN,NaN
1135,UMUC1,TMCC2,32.552,32.475,N,N,N,NaN,NaN
1136,HT8,TMCC2,29.491,29.46,N,N,N,NaN,NaN


In [5]:
combined_RNAseq_df

,baseMean,log2FoldChange,lfcSE,pvalue,padj,GeneSymbol,gene_type,Sample
0,7498.789751,-2.335216,0.053240,0.000000,0.0,S100A14,protein_coding,UMUC1-HT8
1,4147.558405,-3.142345,0.073502,0.000000,0.0,PBX1,protein_coding,UMUC1-HT8
2,9536.176009,1.998321,0.048639,0.000000,0.0,EPHX1,protein_coding,UMUC1-HT8
3,6757.812006,-3.355842,0.055428,0.000000,0.0,SCCPDH,protein_coding,UMUC1-HT8
4,3632.633727,-3.132228,0.076805,0.000000,0.0,SEMA3F,protein_coding,UMUC1-HT8
...,...,...,...,...,...,...,...,...
70294,0.551443,-0.003688,0.221832,0.384585,NaN,AC009235.4,processed_pseudogene,647V-1C2
70295,0.655140,0.004744,0.221592,0.284238,NaN,CD24L4,processed_pseudogene,647V-1C2
70296,0.924396,-0.000465,0.221418,0.416971,NaN,RP11-576C2.1,transcribed_processed_pseudogene,647V-1C2
70297,4.840799,-0.012903,0.220905,0.891264,NaN,RP11-424G14.1,lincRNA,647V-1C2


In [136]:
from __future__ import annotations

import pandas as pd


def summarize_qpcr_rnaseq(
    qpcr_df: pd.DataFrame,
    rnaseq_df: pd.DataFrame,
    *,
    ct_threshold: float = 35.0,
    sd_threshold: float = 0.5,
    expected_reps: int = 2,
    gene_map: dict[str, str] | None = None,
    sample_map: dict[str, str] | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Returns
    -------
    qpcr_summary
        One row per qPCR sample and target, including replicate QC and the
        matching RNA-seq baseMean for that cell line.

    rnaseq_summary
        One row per qPCR gene, with one baseMean column per RNA-seq cell line.

    overall_summary
        One row per gene, with mean qPCR statistics and mean baseMean across
        cell lines.
    """
    gene_map = gene_map or {}
    sample_map = sample_map or {}

    required_qpcr = {"Sample Name", "Target Name", "CT"}
    required_rnaseq = {"Sample", "GeneSymbol", "baseMean"}

    missing_qpcr = required_qpcr - set(qpcr_df.columns)
    missing_rnaseq = required_rnaseq - set(rnaseq_df.columns)

    if missing_qpcr:
        raise ValueError(f"Missing qPCR columns: {sorted(missing_qpcr)}")
    if missing_rnaseq:
        raise ValueError(f"Missing RNA-seq columns: {sorted(missing_rnaseq)}")
    if expected_reps < 1:
        raise ValueError("expected_reps must be at least 1.")

    qpcr = qpcr_df.copy()
    rnaseq = rnaseq_df.copy()

    qpcr["CT"] = pd.to_numeric(qpcr["CT"], errors="coerce")
    rnaseq["baseMean"] = pd.to_numeric(rnaseq["baseMean"], errors="coerce")

    # Convert optional failure columns to Boolean flags.
    if "NOAMP" in qpcr.columns:
        qpcr["_noamp"] = (
            qpcr["NOAMP"].fillna("N").astype(str).str.upper().eq("Y")
        )
    else:
        qpcr["_noamp"] = False

    if "EXPFAIL" in qpcr.columns:
        qpcr["_expfail"] = (
            qpcr["EXPFAIL"].fillna("N").astype(str).str.upper().eq("Y")
        )
    else:
        qpcr["_expfail"] = False

    # Failed wells are counted for QC but excluded from Ct calculations.
    valid_well = ~(qpcr["_noamp"] | qpcr["_expfail"])
    qpcr["_valid_ct"] = qpcr["CT"].where(valid_well)

    # Summarize qPCR replicates.
    qpcr_summary = (
        qpcr.groupby(["Sample Name", "Target Name"], as_index=False)
        .agg(
            mean_ct=("_valid_ct", "mean"),
            ct_sd=("_valid_ct", "std"),
            max_ct=("_valid_ct", "max"),
            observed_reps=("CT", "size"),
            valid_reps=("_valid_ct", "count"),
            noamp=("_noamp", "any"),
            expfail=("_expfail", "any"),
        )
    )

    qpcr_summary["missing_reps"] = (
        expected_reps - qpcr_summary["valid_reps"]
    ).clip(lower=0)

    # high_ct means at least one valid replicate exceeded the threshold.
    qpcr_summary["high_ct"] = qpcr_summary["max_ct"].gt(ct_threshold)

    qpcr_summary["high_sd"] = (
        qpcr_summary["ct_sd"].gt(sd_threshold).fillna(False)
    )

    qpcr_summary["amplification_failure"] = (
        qpcr_summary["noamp"]
        | qpcr_summary["expfail"]
        | qpcr_summary["valid_reps"].lt(expected_reps)
    )

    # Map qPCR target names to RNA-seq gene symbols.
    qpcr_summary["GeneSymbol"] = (
        qpcr_summary["Target Name"]
        .map(gene_map)
        .fillna(qpcr_summary["Target Name"])
    )

    # Map qPCR sample names to RNA-seq cell-line names.
    qpcr_summary["RNAseq Sample"] = (
        qpcr_summary["Sample Name"]
        .map(sample_map)
        .fillna(qpcr_summary["Sample Name"])
    )

    # Keep only genes included in the qPCR experiment.
    qpcr_genes = qpcr_summary["GeneSymbol"].dropna().unique()

    rnaseq_long = (
        rnaseq.loc[
            rnaseq["GeneSymbol"].isin(qpcr_genes),
            ["GeneSymbol", "Sample", "baseMean"],
        ]
        .dropna(subset=["GeneSymbol", "Sample"])
        .drop_duplicates()
    )

    # Do not silently average duplicate baseMean values.
    duplicates = rnaseq_long.duplicated(
        subset=["GeneSymbol", "Sample"],
        keep=False,
    )

    if duplicates.any():
        duplicate_rows = rnaseq_long.loc[
            duplicates, ["GeneSymbol", "Sample", "baseMean"]
        ].sort_values(["GeneSymbol", "Sample"])

        raise ValueError(
            "Multiple baseMean values were found for the same gene and "
            "RNA-seq sample:\n"
            f"{duplicate_rows.to_string(index=False)}"
        )

    # Add the cell-line-specific baseMean to the qPCR summary.
    qpcr_summary = qpcr_summary.merge(
        rnaseq_long.rename(columns={"Sample": "RNAseq Sample"}),
        on=["GeneSymbol", "RNAseq Sample"],
        how="left",
        validate="many_to_one",
    )

    qpcr_summary = qpcr_summary[
        [
            "Sample Name",
            "Target Name",
            "GeneSymbol",
            "RNAseq Sample",
            "mean_ct",
            "ct_sd",
            "max_ct",
            "baseMean",
            "observed_reps",
            "valid_reps",
            "missing_reps",
            "high_ct",
            "high_sd",
            "noamp",
            "expfail",
            "amplification_failure",
        ]
    ]

    # RNA-seq summary: genes as rows and cell lines as columns.
    rnaseq_summary = (
        rnaseq_long.pivot(
            index="GeneSymbol",
            columns="Sample",
            values="baseMean",
        )
        .reset_index()
    )
    rnaseq_summary.columns.name = None

    # Overall quick-glance summary.
    # mean_ct and ct_sd are averages of the sample-level qPCR summaries.
    qpcr_overall = (
        qpcr_summary.groupby(
            ["Target Name", "GeneSymbol"],
            as_index=False,
        )
        .agg(
            mean_ct=("mean_ct", "mean"),
            ct_sd=("ct_sd", "mean"),
        )
    )

    # Here baseMean is intentionally averaged across RNA-seq cell lines.
    rnaseq_overall = (
        rnaseq_long.groupby("GeneSymbol", as_index=False)
        .agg(baseMean=("baseMean", "mean"))
    )

    overall_summary = qpcr_overall.merge(
        rnaseq_overall,
        on="GeneSymbol",
        how="left",
        validate="many_to_one",
    )

    return qpcr_summary, rnaseq_summary, overall_summary

In [137]:
qpcr_summary, rnaseq_summary, overall_summary = summarize_qpcr_rnaseq(
    combined_qPCR_df,
    combined_RNAseq_df,
    ct_threshold=35,
    sd_threshold=0.5
)

In [144]:
gene_of_interest = 'HEATR3'
qpcr_summary[qpcr_summary['Target Name']==gene_of_interest]

,Sample Name,Target Name,GeneSymbol,RNAseq Sample,mean_ct,ct_sd,max_ct,baseMean,observed_reps,valid_reps,missing_reps,high_ct,high_sd,noamp,expfail,amplification_failure
28,1C2,HEATR3,HEATR3,1C2,26.0010,0.019799,26.015,NaN,2,2,0,False,False,False,False,False
96,2F2-EARLY,HEATR3,HEATR3,2F2-EARLY,25.9145,0.021920,25.930,NaN,2,2,0,False,False,False,False,False
164,2F2-LATE,HEATR3,HEATR3,2F2-LATE,26.6870,0.053740,26.725,NaN,2,2,0,False,False,False,False,False
232,647V,HEATR3,HEATR3,647V,26.6320,0.137179,26.729,NaN,2,2,0,False,False,False,False,False
300,HT8,HEATR3,HEATR3,HT8,26.0990,0.069296,26.148,NaN,2,2,0,False,False,False,False,False
368,RT112,HEATR3,HEATR3,RT112,25.9640,0.087681,26.026,NaN,2,2,0,False,False,False,False,False
436,UMUC1,HEATR3,HEATR3,UMUC1,27.2240,0.264458,27.411,NaN,2,2,0,False,False,False,False,False


In [145]:
rnaseq_summary[rnaseq_summary['GeneSymbol']==gene_of_interest]

,GeneSymbol,647V-1C2,RT112-2F2,UMUC1-HT8
27,HEATR3,109.584279,735.040718,1391.337802


In [146]:
overall_summary[overall_summary['GeneSymbol']==gene_of_interest]

,Target Name,GeneSymbol,mean_ct,ct_sd,baseMean
28,HEATR3,HEATR3,26.360214,0.093439,745.320933
